### Titanic - cardinality

Łukasz Całka <br>
Uczenie Maszynowe

In [1]:
import pandas as pd

Wczytanie danych:

In [2]:
kolumny = [
    "pclass", "survived", "name", "sex", "age", "sibsp",
    "parch", "ticket", "fare", "cabin", "embarked",
    "boat", "body", "home.dest"
]

df = pd.read_csv(
    "Zbiór danych Titanic.arff",
    skiprows=17,
    header=None,
    names=kolumny,
    na_values="?"
)

In [3]:
display(df.head(20))

print(len(df))

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
5,1,1,"Anderson, Mr. Harry",male,48.0000,0,0,19952,26.5500,E12,S,3,NaN,"New York, NY"
6,1,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1,0,13502,77.9583,D7,S,10,NaN,"Hudson, NY"
7,1,0,"Andrews, Mr. Thomas Jr",male,39.0000,0,0,112050,0.0000,A36,S,NaN,NaN,"Belfast, NI"
8,1,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2,0,11769,51.4792,C101,S,D,NaN,"Bayside, Queens, NY"
9,1,0,"Artagaveytia, Mr. Ramon",male,71.0000,0,0,PC 17609,49.5042,NaN,C,NaN,22.0,"Montevideo, Uruguay"


1309


Liczebnośc etykiet dla danej zmiennej jakościowej:

In [4]:
zmienne_kategoryczne = ["pclass", "survived", "name", "sex", "ticket", "cabin", "embarked", "boat", "home.dest"]

for kolumna in zmienne_kategoryczne:
    print(f"Liczba etykiet zmiennej {kolumna}: {len(df[kolumna].unique())}")


Liczba etykiet zmiennej pclass: 3
Liczba etykiet zmiennej survived: 2
Liczba etykiet zmiennej name: 1307
Liczba etykiet zmiennej sex: 2
Liczba etykiet zmiennej ticket: 929
Liczba etykiet zmiennej cabin: 187
Liczba etykiet zmiennej embarked: 4
Liczba etykiet zmiennej boat: 28
Liczba etykiet zmiennej home.dest: 370


Liczba pasażerów:


In [5]:
print('Liczba wszystkich pasażerów: {}'.format(len(df.index.unique())))

Liczba wszystkich pasażerów: 1309


Zmienne jakościowe w zbiorze mają bardzo różną kardynalność. Możemy je podzielić na dwie grupy. Zmienne o dużej kardynalności, czyli dużej ilości unikalnych etykiet. W naszym zbiorze są to na pewno name, ticket, cabin oraz home.dest. Są one praktycznie unikalne dla każdego pasażera (który je posiada, sporo jest braków danych), więc kategorii jest bardzo dużo. Są one nieprkatyczne z punktu widzenia algorytmów uczenia maszynowego, gdyż bardzo ciężko je zakodować. Drugą grupą są zmienne o niskiej kardynalności, u nas są to pclass, survived, sex i embarked posiadające ledwie po kilka kategorii oraz boat z 28 etykietami, co wciąż wydaje się realną do zakodowania ilością. Te zmienne jesteśmy w stanie łatwo przerobić na format wygodny dla algorytmów (np. macierz z 0 i 1), co pozwala skutecznie wykorzystać je przy uczeniu modelu.
<br>
Porównując wynik funkcji len na indeksie (1309) oraz funkcji len na unikalnych imionach (1307) dochodzimy do wniosku, że na statku istaniały dwie pary osób z tym samym imieniem i nazwiskiem.

Liczba unikalnych kabin w postaci NumPy array:

In [6]:
kabiny_unikalne = df['cabin'].unique().to_numpy()
print(f"Typ zwracanej wartości: {type(kabiny_unikalne)}")
print(f"Liczba unikalnych etykiet kabin: {len(kabiny_unikalne)}")


Typ zwracanej wartości: <class 'numpy.ndarray'>
Liczba unikalnych etykiet kabin: 187


Zredukowana zmienna dla kabin:

In [7]:
df["CabinReduced"] = df["cabin"].astype(str).str[0]

display(df[["cabin", "CabinReduced"]].head(20))


,cabin,CabinReduced
0,B5,B
1,C22 C26,C
2,C22 C26,C
3,C22 C26,C
4,C22 C26,C
5,E12,E
6,D7,D
7,A36,A
8,C101,C
9,NaN,NaN


Liczba etykiet zmiennej CabinReduced:

In [8]:
print(f"Liczba etykiet zmiennej {"CabinReduced"}: {len(df["CabinReduced"].unique())}")
zredukowane_kabiny_unikalne = df['CabinReduced'].unique().to_numpy()
redukcja = ((len(kabiny_unikalne)-len(zredukowane_kabiny_unikalne))/len(kabiny_unikalne)) * 100
print(f"Zredukowano kardynalność zmiennej opisującej kabiny o {redukcja:.2f}%")

Liczba etykiet zmiennej CabinReduced: 9
Zredukowano kardynalność zmiennej opisującej kabiny o 95.19%


Zredukowaliśmy tę zmienną, ponieważ cechowała się bardzo wysoką kardynalnością, aż 187 unikalnych wartości. Wartości te jednak miały podobną budowę X99, co pozwoliło, dość łatwo skonwertować je do tylko pierwszych liter, które niosą za sobą najistotniejszą informację o pokładzie, na którym znajdowała się kabina. Pozwala to nam zakodować ją i wykorzystać w dalszej analizie oraz przy tworzeniu algorytmów. Gdyby zmienna pozostała w oryginalnej formie - 187 wartości, zakodowanie jej w użyteczny z perspektywy ML format byłoby bardzo trudne. Taka zmiana wiąże się oczywiście z utratą części danych, w tym wypadku numeru kabiny. Jeśli numer kabiny, a więc lokalizacja z danej strony statku miała wpływ na przeżywalności zatracimy ją. Musimy się na to zgodzić, aby dopasować zbiór konieczne są pewne ustępstwa.

W zadaniu wykorzystano narzędzia generatywnej sztucznej inteligencji w celu szybkiego wyekstraktowania nazw kolumn z pliku ze zbiorem danych a także do poprawnego zredagowania wniosków obserwowanych w zbiorze danych.